<a href="https://colab.research.google.com/github/Phavouredphavour/Pizza-Sales-Analysis-/blob/main/curriculum/phase-2b-sql/weeks-01-08-teaching/week-03-joins/01-wednesday/lecture-materials/week-03-wed-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — JOINs: Connecting Tables
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Write an `INNER JOIN` to combine two tables on a shared key, using table aliases
- Filter and group **across both tables** in a single query — a `WHERE` on one table and a `GROUP BY` on the other
- Recognise what an `INNER JOIN` silently discards, and reach for a `LEFT JOIN` when those rows matter

For two weeks every query you wrote read from exactly **one** table. That is a real
ceiling: the most interesting business questions live in the gap *between* tables.
Today you learn how to close that gap.

### Run this first

The setup cell below loads all 8 Olist tables into a SQLite database and connects the
`%%sql` magic to it. It is the same cell as Weeks 1 and 2 — run it once, wait for
`Database ready.`, and leave it alone.

In [1]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

Mounted at /content/drive
Searching your Google Drive for phase-2-python-sql.zip ...
Unzipping phase-2-python-sql.zip ...
Data folder: /content/olist_data/phase-2-python-sql
Loaded orders: 99,441 rows
Loaded customers: 99,441 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded product_category_translation: 71 rows

Database ready.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.8/607.8 kB 8.2 MB/s eta 0:00:00


## Why this matters

Here is a question the Olist commercial team genuinely asks: **which Brazilian states
do our delivered orders come from?** You already know every clause you need to answer
it — `WHERE`, `GROUP BY`, `COUNT`, `ORDER BY`. And you cannot answer it.

The reason is that the answer is split across two tables. `orders` has all 99,441
orders and their status, but it does **not** hold a state — it holds a
`customer_id`, a 32-character code. `customers` has the `customer_state`, but it has
no idea which orders belong to whom. Each table holds half the answer.

A **JOIN** is how SQL stitches the halves together: you tell the database which column
in one table points at which column in the other, and it hands you back rows that
carry columns from both. Every serious analytical query you write from here on will
have a JOIN in it.

## 1. `INNER JOIN` — matching rows on a shared key

An `INNER JOIN` walks through the left table and, for each row, looks for rows in the
right table where the `ON` condition is true. When it finds a match it emits a single
combined row carrying the columns of both. `orders.customer_id` and
`customers.customer_id` are the shared key here — the same value identifies the same
shopper in both tables.

Think of it as a lookup in a spreadsheet: `orders` is your main sheet, `customer_id`
is the value you look up, and `customers` is the reference sheet you pull the state
back from. SQL just does it for all 99,441 rows at once instead of dragging a formula
down a column.

Two habits to build immediately. First, `JOIN` and `INNER JOIN` mean exactly the same
thing in SQLite — the word `INNER` is optional and most analysts drop it. Second,
**give every table a short alias** (`orders o`, `customers c`) and prefix every column
with it. It is not decoration: both tables have a `customer_id` column, and without
the prefix SQLite cannot tell which one you mean.

In [2]:
%%sql
-- Each row now carries columns from BOTH tables: order_id and order_status come
-- from orders, customer_state and customer_city come from customers.
SELECT o.order_id, o.order_status, c.customer_state, c.customer_city
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
LIMIT 10

,order_id,order_status,customer_state,customer_city
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,SP,sao paulo
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,BA,barreiras
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,GO,vianopolis
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,RN,sao goncalo do amarante
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,SP,santo andre
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,PR,congonhinhas
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,RS,santa rosa
7,6514b8ad8028c9f2cc2374ded245783f,delivered,RJ,nilopolis
8,76c6e866289321a7c93b82b54852dc33,delivered,RS,faxinalzinho
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,SP,sorocaba


## 2. How many rows does a join return?

This is the question to ask yourself before trusting *any* joined result, and the
answer is never automatically "the same as before". A join returns **one row per
matching pair**. If a customer had three orders you would see that customer's state
three times — once per order. If an order's `customer_id` matched nothing in
`customers`, an `INNER JOIN` would drop that order entirely, and your count would
quietly shrink.

Olist's `orders` and `customers` tables happen to be a clean **one-to-one** pair:
99,441 rows each, and every order's `customer_id` appears exactly once in `customers`.
So this particular join neither multiplies nor loses anything, and the count below
comes back identical to the row count of `orders` on its own. Confirming that is a
one-line sanity check, and it is worth doing every time — later today you will meet a
join where the number does **not** come back the same.

In [3]:
%%sql
-- Sanity check: a 1:1 join must not change the row count.
SELECT COUNT(*) AS rows_after_join   -- Expected: 99,441 — identical to the orders table
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id;

,rows_after_join
0,99441


## 3. Filtering and grouping across both tables

Once two tables are joined, SQL stops caring which table a column originally came
from — the joined result behaves like one wide table, and every clause you learned in
Weeks 1 and 2 works on it unchanged. That is the real payoff of a join.

Look at what the query below does: it filters on a column from `orders`
(`WHERE o.order_status = 'delivered'`) and groups by a column from `customers`
(`GROUP BY c.customer_state`), in one pass. Neither table could answer this alone.
The clause order is exactly as before — `FROM` → `JOIN` → `WHERE` → `GROUP BY` →
`ORDER BY` → `LIMIT` — with `JOIN` slotting in right after `FROM`.

The result is the answer to the question we opened with, and it is a stark one: São
Paulo alone accounts for more delivered orders than the next three states combined.

In [4]:
%%sql
-- Delivered orders by customer state: filter on orders, group by customers.
-- Expected top 5: SP 40,501 | RJ 12,350 | MG 11,354 | RS 5,345 | PR 4,923
SELECT c.customer_state, COUNT(*) AS order_count
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY order_count DESC
LIMIT 5;

,customer_state,order_count
0,SP,40501
1,RJ,12350
2,MG,11354
3,RS,5345
4,PR,4923


## 4. Joining a fact table to a lookup table — sellers and their revenue

`orders`/`customers` was one row to one row. The far more common shape in a real
database is **many-to-one**: a big table of events joined to a small table that
describes something. `order_items` holds 112,650 line items and records which
`seller_id` fulfilled each one; `sellers` holds just 3,095 rows, one per seller, with
the state they operate from.

Joining them attaches each seller's state to every line item they sold. Now
`GROUP BY oi.seller_id` collapses those items back down to one row per seller, and the
aggregates tell us how much each one actually sold. Note that grouping by
`s.seller_state` alongside the id is safe here — a seller has exactly one state, so it
adds no new groups; it just carries the state through to the output.

The top seller turns out to have moved **R$229,472.63** of merchandise across 1,156
line items — a genuine standout in a marketplace of 3,095 sellers.

In [5]:
%%sql
-- Top 10 sellers by product revenue, with the state each one ships from.
-- Expected top row: R$229,472.63 across 1,156 items (an SP seller).
SELECT oi.seller_id, s.seller_state,
       COUNT(*) AS items_sold,
       ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY oi.seller_id, s.seller_state
ORDER BY total_revenue DESC
LIMIT 10;

,seller_id,seller_state,items_sold,total_revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1156,229472.63
1,53243585a1d6dc2643021fd1853d8905,BA,410,222776.05
2,4a3ca9315b744ce9f8e9374361493884,SP,1987,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,586,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,SP,1364,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,SP,340,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,1551,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,SP,1171,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,1428,138968.55
9,955fee9216a65b617aa5c0531780ce60,SP,1499,135171.70


## 5. Rolling up one level — revenue by seller state

Same join, one change: group by `s.seller_state` instead of the seller id, and 3,095
sellers collapse into a couple of dozen states. This is the move that turns a long
operational list into an executive summary.

One detail matters a great deal here. We want two different things per state — how
much revenue, and how many sellers produced it — and they need different counters.
`SUM(oi.price)` is correct at line-item grain, because every line item's price should
be added exactly once. But `COUNT(*)` would count **line items**, not sellers, and
report tens of thousands. `COUNT(DISTINCT oi.seller_id)` is what you want: it counts
each seller once no matter how many items they sold. Getting this distinction wrong is
the most common way a joined report ends up confidently reporting the wrong number.

Read the result as a supply-side map: SP is not merely the biggest customer state, it
is overwhelmingly the biggest *seller* state too, with R$8.75m of product revenue —
close to seven times the runner-up.

In [6]:
%%sql
-- Where does Olist's supply come from? Seller count and revenue per state.
-- Expected top 5: SP 1,849 sellers / 8,753,396.21 | PR 349 / 1,261,887.21
--                 MG 244 / 1,011,564.74 | RJ 171 / 843,984.22 | SC 190 / 632,426.07
SELECT s.seller_state,
       COUNT(DISTINCT oi.seller_id) AS seller_count,
       ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY s.seller_state
ORDER BY total_revenue DESC
LIMIT 8;

,seller_state,seller_count,total_revenue
0,SP,1849,8753396.21
1,PR,349,1261887.21
2,MG,244,1011564.74
3,RJ,171,843984.22
4,SC,190,632426.07
5,RS,129,378559.54
6,BA,19,285561.56
7,DF,30,97749.48


## Going deeper — what an `INNER JOIN` throws away, and how `LEFT JOIN` catches it

Every `INNER JOIN` is also a filter, and that is the dangerous part: it keeps only rows
that found a match, and it discards the rest **without saying a word**. No error, no
warning, just a slightly smaller answer than the truth.

A `LEFT JOIN` changes the deal. It keeps *every* row from the left table whether or not
the right table had a match; where there was no match, the right table's columns come
back as `NULL`. That gives you a way to see the discarded rows instead of losing them:
left-join the two tables, then keep only the rows where the right side is `NULL` — by
definition, exactly the rows an inner join would have dropped.

Note the `IS NULL`, not `= NULL`. `NULL` means "unknown", and comparing anything to an
unknown yields unknown rather than true, so `= NULL` matches nothing at all — a silent
zero rather than an error. Run this and you find **775 orders that have no line items
at all**, most of them canceled or unavailable orders that never made it to
fulfilment. Every revenue query built on an inner join to `order_items` has been
quietly excluding them. Tomorrow we build on exactly this.

In [7]:
%%sql
-- Keep ALL orders, then isolate the ones order_items had no match for.
SELECT COUNT(*) AS orders_without_items   -- Expected: 775
FROM orders o
LEFT JOIN order_items oi ON o.order_id = oi.order_id
WHERE oi.order_id IS NULL                 -- IS NULL, never = NULL;

,orders_without_items
0,775


## Common mistakes

**Mistake 1 — losing the `ON`, or forgetting the alias prefix.** Omit the join
condition and SQLite does not object; it pairs every left row with every right row.
Here that is 99,441 × 99,441 ≈ **9.9 billion rows**, which will hang your notebook
rather than error. And because `customer_id` exists in both tables, selecting it
unqualified fails outright with `ambiguous column name: customer_id`. Alias every
table, prefix every column, always write the `ON`.

**Mistake 2 — `COUNT(*)` after a one-to-many join.** This one is subtle enough to ship
to production. `order_items` holds one row *per line item*, so an order with three
items becomes three rows the moment you join it. `COUNT(*)` then counts line items
while the column heading says orders: 112,650 instead of the 98,666 orders that
actually have items. The fix is `COUNT(DISTINCT o.order_id)`, which counts each order
once regardless of how many rows the join produced. Before writing any `COUNT` over a
join, ask yourself: *what is one row in this result?*

In [8]:
%%sql
-- ── COMMON MISTAKE 1: no ON condition, and unqualified columns ──────
-- WRONG — no join condition. SQLite silently pairs EVERY order with EVERY
-- customer: 99,441 × 99,441 ≈ 9.9 BILLION rows. Never run this.
--   SELECT o.order_id, c.customer_state FROM orders o, customers c
--
-- WRONG — customer_id exists in BOTH tables, so this is ambiguous. SQLite
-- raises: "ambiguous column name: customer_id"
--   SELECT customer_id, customer_state
--   FROM orders o JOIN customers c ON o.customer_id = c.customer_id
--
-- CORRECT — state the ON condition, and prefix every column with its alias:
SELECT o.order_id, o.customer_id, c.customer_state
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
LIMIT 5

,order_id,customer_id,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,SP


In [9]:
%%sql
-- ── COMMON MISTAKE 2: COUNT(*) after a one-to-many join ─────────────
-- WRONG — counts LINE ITEMS while calling them orders (returns 112,650):
--   SELECT COUNT(*) AS order_count
--   FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
-- CORRECT — COUNT(DISTINCT ...) counts each order once. Both numbers are
-- shown side by side so you can see how far apart they are:
SELECT COUNT(DISTINCT o.order_id) AS orders_with_items,   -- Expected: 98,666
       COUNT(*)                   AS rows_after_join      -- Expected: 112,650
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id

,orders_with_items,rows_after_join
0,98666,112650


## Mini-challenge — your turn

⏱ ~5–10 minutes

The regional manager for Rio Grande do Sul wants one number: **how many delivered
orders came from customers in `RS`?**

You need `orders` (for the status) and `customers` (for the state), so this is a join
plus a two-condition `WHERE`. Alias both tables and prefix every column. Name your
output column `rs_delivered` so the result reads clearly.

**Expected:** 5,345 — which you can check against section 3, where `RS` sat fourth in
the delivered-orders-by-state ranking.

*Stretch, if you finish early:* swap `COUNT(*)` for
`COUNT(DISTINCT o.order_id)` and confirm the number does not move. Why not? (Because
`orders`→`customers` is one-to-one, so nothing fanned out — the check that section 2
taught you to run.)

In [19]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT COUNT(*) AS rs_delivered
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND c.customer_state = 'RS'

,rs_delivered
0,5345


## Session Summary

| Clause / idea | What it does | Example |
|---|---|---|
| `JOIN ... ON` | combine two tables on a shared key | `FROM orders o JOIN customers c ON o.customer_id = c.customer_id` |
| `INNER JOIN` | identical to `JOIN`; keeps only matched rows | `INNER JOIN customers c ON ...` |
| table alias | short name for a table, prefixes its columns | `orders o` → `o.order_status` |
| `WHERE` on a join | filters the combined rows (can use either table) | `WHERE o.order_status = 'delivered'` |
| `GROUP BY` on a join | groups by a column from the *other* table | `GROUP BY c.customer_state` |
| `COUNT(DISTINCT col)` | counts each entity once after a one-to-many join | `COUNT(DISTINCT oi.seller_id)` |
| `LEFT JOIN` | keeps every left row; unmatched right columns become `NULL` | `LEFT JOIN order_items oi ON ...` |
| `IS NULL` | finds the unmatched rows (never `= NULL`) | `WHERE oi.order_id IS NULL` |

**The three questions to ask about every join:** which column links the two tables?
Is the relationship one-to-one or one-to-many? And what does one row of the result
represent — because that determines whether `COUNT(*)` is telling you the truth.

---
**Coming up Thursday**: we go properly into `LEFT JOIN` and NULL handling — inspecting
those 775 order-less orders to see what statuses they carry — and then stretch to
**three-table joins**, pulling `orders`, `customers` and `order_payments` into a single
query. You will finish with a group exercise set covering state-level counts, average
payment values, seller distribution, and review scores.